# 📖 Novel-TUI — Servidor LLM Remoto (GPU T4 en Google Colab)

Este cuaderno ejecuta **KoboldCpp** con aceleración CUDA en GPU T4 y el modelo **Magnum-12B-v2** (GGUF Q4_K_M 100% sin censura para literatura y R-18).

### ⚡ Ventajas:
1. **0% Censura:** Basado en Mistral-Nemo 12B, entrenado específicamente para ficción adulta, narrativa literaria y cero rechazos morales.
2. **Persistencia en Google Drive:** El modelo se descarga una sola vez en tu Drive (`/NovelTUI_Models/`) y en los próximos arranques inicia en 5 segundos.
3. **Túnel Cloudflare Gratuito:** Genera una URL pública `https://*.trycloudflare.com/v1` compatible con la API de OpenAI para Novel-TUI.

In [ ]:
#@title 🚀 Iniciar Servidor KoboldCpp con GPU y Google Drive (Magnum 12B Uncensored)
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR = '/content/drive/MyDrive/NovelTUI_Models'
MODEL_PATH = os.path.join(DRIVE_DIR, 'magnum-12b-v2-Q4_K_M.gguf')
MODEL_URL = 'https://huggingface.co/bartowski/magnum-12b-v2-GGUF/resolve/main/magnum-12b-v2-Q4_K_M.gguf'
KOBOLD_URL = 'https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64'

!mkdir -p /content/novel-llm
!mkdir -p "{DRIVE_DIR}"
%cd /content/novel-llm

# 1. Descargar KoboldCpp si no existe
if not os.path.exists('koboldcpp_linux') or os.path.getsize('koboldcpp_linux') < 1000000:
    print('📥 Descargando KoboldCpp...')
    !wget -q -c {KOBOLD_URL} -O koboldcpp_linux
    !chmod +x koboldcpp_linux

# 2. Descargar o enlazar modelo Magnum 12B Uncensored
if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 5000000000:
    print('⚡ Modelo Magnum 12B encontrado en Google Drive! Enlazando...')
    !ln -sf "{MODEL_PATH}" model.gguf
else:
    print('📥 Descargando Magnum-12B-v2 a Google Drive (7.4 GB - 100% Uncensored)...')
    !wget -c "{MODEL_URL}" -O "{MODEL_PATH}"
    !ln -sf "{MODEL_PATH}" model.gguf

# 3. Iniciar KoboldCpp con todas las capas en GPU T4 y túnel remoto
print('🚀 Iniciando servidor KoboldCpp con GPU y Túnel Cloudflare...')
!./koboldcpp_linux --model model.gguf --usecuda 0 mmq --gpulayers 999 --contextsize 8192 --remotetunnel
